# Solution — tune a model end to end

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RSEED = 42

penguins = pd.read_csv("../data/penguins.csv").dropna(subset=["sex"])

X = penguins.drop(columns="sex")
y = penguins["sex"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RSEED, stratify=y
)

numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy="most_frequent"),
                            OneHotEncoder(handle_unknown="ignore"))

preprocessor = make_column_transformer(
    (numeric, make_column_selector(dtype_include="number")),
    (categorical, make_column_selector(dtype_exclude="number")),
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

print(f"{len(X_train)} training penguins, {len(X_test)} held out")

## Task 1 — find the tunable names

In [ ]:
hgb = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(random_state=RSEED)),
])

for name, value in sorted(hgb.get_params().items()):
    if name.startswith("classifier__"):
        print(f"  {name:42s} = {value}")

The ones that matter most for a boosted model are `learning_rate` (how big a step each tree
takes), `max_leaf_nodes` (how complex each tree may be) and `max_iter` (how many trees).
`learning_rate` and `max_iter` trade off against each other — a smaller step needs more
trees.

## Task 2 — a baseline

In [ ]:
baseline_scores = cross_val_score(hgb, X_train, y_train, cv=5, scoring="accuracy")
baseline = baseline_scores.mean()

print(f"default HistGradientBoosting: {baseline:.4f} ± {baseline_scores.std():.4f}")

## Task 3 — grid search

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__learning_rate": [0.01, 0.05, 0.1, 0.5],
    "classifier__max_leaf_nodes": [5, 15, 31, 63],
}

combinations = 4 * 4
print(f"{combinations} combinations × 5 folds = {combinations * 5} fits\n")

grid = GridSearchCV(hgb, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print(f"best params : {grid.best_params_}")
print(f"best cv     : {grid.best_score_:.4f}")
print(f"baseline    : {baseline:.4f}")
print(f"improvement : {grid.best_score_ - baseline:+.4f}")

## Task 4 — randomized search

In [ ]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "classifier__learning_rate": loguniform(1e-2, 1e0),
    "classifier__max_leaf_nodes": randint(5, 64),
}

rand = RandomizedSearchCV(hgb, param_distributions, n_iter=30, cv=5,
                          scoring="accuracy", random_state=RSEED, n_jobs=-1)
rand.fit(X_train, y_train)

print(f"grid   : {grid.best_score_:.4f} from {combinations * 5} fits")
print(f"random : {rand.best_score_:.4f} from {30 * 5} fits")
print(f"\nrandom best params: {rand.best_params_}")

Random search costs about twice as much here (150 fits against 80) because we chose
`n_iter=30`. The budgets are ours to set — the honest comparison is score per fit, and on a
two-parameter space a grid is perfectly reasonable. Random search earns its keep once there
are four or five parameters.

## Task 5 — how confident are you?

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)

tuner = GridSearchCV(hgb, param_grid, cv=inner_cv, scoring="accuracy", n_jobs=-1)
nested = cross_validate(tuner, X_train, y_train, cv=outer_cv, scoring="accuracy")

# the same treatment for the untuned baseline, so the comparison is like for like
baseline_nested = cross_validate(hgb, X_train, y_train, cv=outer_cv, scoring="accuracy")

print(f"tuned    : {nested['test_score'].mean():.4f} ± {nested['test_score'].std():.4f}")
print(f"baseline : {baseline_nested['test_score'].mean():.4f} ± {baseline_nested['test_score'].std():.4f}")

### Would you claim the tuned model beats the baseline?

**No. And look at *why* — this result is sharper than a simple "no".**

```
tuned    : 0.9038 ± 0.0341
baseline : 0.9158 ± 0.0318
```

Under nested cross-validation the **tuned model scores lower than the untuned one**. Yet
`grid.best_score_` reported 0.9277 against a baseline of 0.9116 — an apparent improvement of
+0.016.

Both numbers are real. They measure different things:

- `best_score_` is the **maximum of sixteen noisy estimates**. Take sixteen measurements of
  the same quantity and keep the largest, and you get a number above the truth. It is biased
  upward by construction, and the bias grows with the size of the grid.
- The nested estimate re-runs the entire selection on each outer fold, so the score is never
  computed on data that influenced the choice.

The intervals overlap heavily, so the honest reading is that **tuning changed nothing here**
— the difference between 0.904 and 0.916 is well inside the fold-to-fold spread of ±0.03.

What you can say: *the defaults were already close to the best this model family can do on
this data, and the apparent +0.016 from tuning was selection bias, not a gain.* That is a
genuinely useful finding — it tells you to spend your next hour on features or a different
model, not on a bigger grid.

**This is the single most important habit in the lesson.** Reporting `best_score_` as a
result is the most common way that honest people overstate their models.

## Stretch — searching preprocessing too

In [ ]:
stretch_grid = {
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_leaf_nodes": [15, 31],
    "preprocessor__pipeline-1__simpleimputer__strategy": ["mean", "median"],
}

stretch = GridSearchCV(hgb, stretch_grid, cv=5, scoring="accuracy", n_jobs=-1)
stretch.fit(X_train, y_train)

results = pd.DataFrame(stretch.cv_results_)
pivot = results.pivot_table(
    index=["param_classifier__learning_rate", "param_classifier__max_leaf_nodes"],
    columns="param_preprocessor__pipeline-1__simpleimputer__strategy",
    values="mean_test_score",
)
pivot.round(4)

Compare the two columns row by row. If the better imputation strategy is the same in every
row, the choice is **independent** of the classifier settings and could have been made
separately. If it flips between rows, the parameters **interact** and have to be searched
together.

That is the real argument for putting preprocessing inside the search rather than fixing it
beforehand — you cannot know which case you are in until you look.

> Note that `HistGradientBoostingClassifier` handles missing values natively, so imputation
> matters less to it than it would to a linear model. Trying this stretch task with
> `LogisticRegression` as the classifier gives a livelier result.